# voxstage on Colab: first real run

Runtime > Change runtime type > **T4 GPU**. Add `GEMINI_API_KEY` in the Secrets panel (key icon, left sidebar) and enable notebook access.

Order matters: **mock smoke test first** (free, checks the plumbing), then real vendors.

In [ ]:
# 1) Get the code. Upload voicebot-engine.zip via the Files panel, or git clone your repo.
import os, zipfile
if os.path.exists('voicebot-engine.zip'):
    zipfile.ZipFile('voicebot-engine.zip').extractall('.')
%cd voicebot-engine
!pip install -q numpy matplotlib scipy
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2) Mock smoke test: no GPU, no key, no cost.
!python -m unittest discover -s tests
!python -m voxstage.make_audio --tts mock --out /content/audio_mock
!python -m voxstage.bench --audio-dir /content/audio_mock --out /content/runs/mock --repeats 3 --warmup 1 --asr-opt time_scale=0.05 --llm-opt time_scale=0.05 --tts-opt time_scale=0.05

In [ ]:
# 3) Install real vendors. MeloTTS Korean needs extra text-frontend packages and can be
#    version-sensitive. If this cell fails, copy the error: the TTS adapter is swappable.
!pip install -q faster-whisper google-genai
!pip install -q git+https://github.com/myshell-ai/MeloTTS.git
!python -m unidic download

In [ ]:
# 4) Key + pinned model. Check Google AI Studio for a Flash-class model that is free *today*
#    and write its exact name here. The model is pinned per run and recorded in meta.json.
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
GEMINI_MODEL = 'PUT-A-FREE-TIER-FLASH-MODEL-NAME-HERE'

# Test audio from the real TTS (clean, ground-truth end of speech in the sidecar json)
!python -m voxstage.make_audio --tts melo --tts-opt device=cuda --out /content/audio_real

In [ ]:
# 5) Small first run: 5 scenarios x 4 repeats = 20 turns (~3-4 min at 8 requests/min).
!python -m voxstage.bench --audio-dir /content/audio_real --out /content/runs/first_real \
  --asr whisper --asr-opt model=large-v3-turbo --asr-opt device=cuda \
  --llm gemini --llm-opt model={GEMINI_MODEL} --llm-opt thinking=off \
  --tts melo --tts-opt device=cuda \
  --rpm 8 --repeats 4 --warmup 3

In [ ]:
# 6) Look at the result, then download runs/first_real (events.jsonl is the raw evidence).
from IPython.display import Image, Markdown, display
display(Markdown(open('/content/runs/first_real/report.md', encoding='utf-8').read()))
display(Image('/content/runs/first_real/latency_hist.png'))
!zip -qr /content/first_real.zip /content/runs/first_real